# Demo: Backtesting with `ForecastingAssistant`

This notebook demonstrates the backtesting workflow:

1. `create_cv()` — Create a cross-validation strategy (deterministic or LLM-assisted)
2. `backtest()` — Run backtesting evaluation
3. `ask(backtest_result=...)` — Explain results with an LLM

**Design principles:**
- `cv` is mandatory in `backtest()` — no hidden defaults
- `create_cv()` absorbs all complexity (deterministic defaults or LLM)
- The returned `TimeSeriesFold` is a standard skforecast object — zero lock-in

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import warnings

import numpy as np
import pandas as pd

from skforecast_ai import ForecastingAssistant

warnings.filterwarnings("ignore")
assistant = ForecastingAssistant()

## Synthetic Data

In [3]:
rng = np.random.default_rng(42)
n = 365
dates = pd.date_range("2022-01-01", periods=n, freq="D")

# Single series with exogenous variable
target = 100 + np.cumsum(rng.normal(0, 1, n)) + 5 * np.sin(2 * np.pi * np.arange(n) / 7)
exog = rng.normal(50, 10, n)

df_single = pd.DataFrame({
    "date": dates,
    "sales": target,
    "promo": exog,
})

# Multi-series (long format)
n_multi = 200
dates_multi = pd.date_range("2022-01-01", periods=n_multi, freq="D")
df_multi = pd.DataFrame({
    "date": np.tile(dates_multi, 3),
    "series_id": ["store_A"] * n_multi + ["store_B"] * n_multi + ["store_C"] * n_multi,
    "revenue": np.concatenate([
        100 + np.cumsum(rng.normal(0, 1, n_multi)),
        200 + np.cumsum(rng.normal(0, 1.5, n_multi)),
        150 + np.cumsum(rng.normal(0, 0.8, n_multi)),
    ]),
})

print(f"df_single: {df_single.shape}")
print(f"df_multi:  {df_multi.shape}")

df_single: (365, 3)
df_multi:  (600, 3)


---
## 1. Basic Backtesting (Deterministic CV)

The simplest workflow: profile → plan → create_cv → backtest.

`create_cv()` with no prompt uses smart deterministic defaults:
- `initial_train_size`: 70% of data
- `refit`: True (refit each fold)
- `fixed_train_size`: False (expanding window)
- `steps`: from plan

In [4]:
# Step 1: Profile and plan
profile = assistant.profile(
    data=df_single, target="sales", date_column="date"
)
plan = assistant.plan(profile, steps=14)

print(f"Forecaster: {plan.forecaster}")
print(f"Estimator:  {plan.estimator}")
print(f"Steps:      {plan.steps}")

Forecaster: ForecasterRecursive
Estimator:  LGBMRegressor
Steps:      14


In [5]:
# Step 2: Generate cross-validation strategy
cv, explanation = assistant.create_cv(profile, plan)

print(cv)
print("")

print(explanation)
print(f"\nTimeSeriesFold object: {cv}")
print(f"  steps:              {cv.steps}")
print(f"  initial_train_size: {cv.initial_train_size}")
print(f"  refit:              {cv.refit}")
print(f"  fixed_train_size:   {cv.fixed_train_size}")

TimeSeriesFold 
Initial train size    = 255,
Steps                 = 14,
Fold stride           = 14,
Overlapping folds     = False,
Window size           = None,
Differentiation       = None,
Refit                 = True,
Fixed train size      = False,
Gap                   = 0,
Skip folds            = None,
Allow incomplete fold = True,
Return all indexes    = False,
Verbose               = False


Using 70% of data (255 observations) for initial training, expanding window, refit every fold, 14-step horizon, 8 folds.

TimeSeriesFold object: ============== 
TimeSeriesFold 
Initial train size    = 255,
Steps                 = 14,
Fold stride           = 14,
Overlapping folds     = False,
Window size           = None,
Differentiation       = None,
Refit                 = True,
Fixed train size      = False,
Gap                   = 0,
Skip folds            = None,
Allow incomplete fold = True,
Return all indexes    = False,
Verbose               = False

  steps:              14
  initial

In [6]:
cv

============== 
TimeSeriesFold 
============== 
Initial train size    = 255,
Steps                 = 14,
Fold stride           = 14,
Overlapping folds     = False,
Window size           = None,
Differentiation       = None,
Refit                 = True,
Fixed train size      = False,
Gap                   = 0,
Skip folds            = None,
Allow incomplete fold = True,
Return all indexes    = False,
Verbose               = False

In [7]:
# Step 3: Run backtesting
result = assistant.backtest(
    data=df_single,
    target="sales",
    cv=cv,
    date_column="date",
)

print("Metrics:")
display(result.metrics)
print(f"\nPredictions shape: {result.predictions.shape}")
display(result.predictions.head(10))

100%|██████████| 8/8 [00:00<00:00, 14.65it/s]

Metrics:


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,2.992479,12.729062,1.016865,0.031984



Predictions shape: (110, 2)


,fold,pred
2022-09-13,0,94.035996
2022-09-14,0,90.509151
2022-09-15,0,86.113216
2022-09-16,0,83.836703
2022-09-17,0,88.117556
2022-09-18,0,91.897009
2022-09-19,0,95.070760
2022-09-20,0,93.275674
2022-09-21,0,88.790373
2022-09-22,0,84.401294


In [8]:
# Inspect the generated code
print(result.code)

import pandas as pd
from lightgbm import LGBMRegressor
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import backtesting_forecaster, TimeSeriesFold
from skforecast.preprocessing import RollingFeatures

# Load data
data = pd.read_csv('data.csv')

data['date'] = pd.to_datetime(data['date'])
data = data.set_index('date')
data = data.asfreq('D')
data = data.sort_index()

window_features = RollingFeatures(
    stats        = ['mean', 'mean'],
    window_sizes = [7, 91],
)

# Create forecaster
forecaster = ForecasterRecursive(
    estimator            = LGBMRegressor(random_state=123, verbose=-1),
    lags                 = [1, 3, 7, 8, 10, 12, 118],
    window_features      = window_features,
    categorical_features = 'auto',
    dropna_from_series   = False,
)

# Cross-validation configuration
cv = TimeSeriesFold(
    steps              = 14,
    initial_train_size = 255,
    refit              = True,
    fixed_train_size   = False,
)

# Run backtest

---
## 2. Custom CV Configuration

Override specific parameters while keeping smart defaults for the rest.

In [9]:
# Fixed training window, no refit, custom initial_train_size
cv_custom, explanation = assistant.create_cv(
    profile, plan,
    initial_train_size=200,
    refit=False,
    fixed_train_size=True,
)

print(explanation)
print(f"\n  initial_train_size: {cv_custom.initial_train_size}")
print(f"  refit:              {cv_custom.refit}")
print(f"  fixed_train_size:   {cv_custom.fixed_train_size}")

Using 55% of data (200 observations) for initial training, fixed window, no refit, 14-step horizon, 12 folds.

  initial_train_size: 200
  refit:              False
  fixed_train_size:   True


In [10]:
result_custom = assistant.backtest(
    data=df_single,
    target="sales",
    cv=cv_custom,
    date_column="date",
)

print("Metrics (fixed window, no refit):")
display(result_custom.metrics)

100%|██████████| 12/12 [00:00<00:00, 841.58it/s]

Metrics (fixed window, no refit):


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,2.946318,12.026144,1.031953,0.032393


---
## 3. Fractional `initial_train_size`

Pass a float between 0 and 1 to use a fraction of the data.

In [11]:
cv_frac, explanation = assistant.create_cv(
    profile, plan,
    initial_train_size=0.8,  # 80% of data
)

print(explanation)
print(f"Resolved initial_train_size: {cv_frac.initial_train_size} "
      f"(80% of {profile.data_profile.n_observations})")

Using 80% of data (292 observations) for initial training, expanding window, refit every fold, 14-step horizon, 6 folds.
Resolved initial_train_size: 292 (80% of 365)


---
## 4. Multi-Series Backtesting

Works the same way — just provide multi-series data.

In [12]:
profile_multi = assistant.profile(
    data=df_multi,
    target="revenue",
    date_column="date",
    series_id_column="series_id",
)
plan_multi = assistant.plan(profile_multi, steps=7)

print(f"Task type:  {plan_multi.task_type}")
print(f"Forecaster: {plan_multi.forecaster}")

Task type:  multi_series
Forecaster: ForecasterRecursiveMultiSeries


In [13]:
cv_multi, explanation = assistant.create_cv(profile_multi, plan_multi)
print(explanation)

Using 78% of data (157 observations) for initial training, expanding window, refit every fold, 7-step horizon, 7 folds.


In [14]:
result_multi = assistant.backtest(
    data=df_multi,
    target="revenue",
    cv=cv_multi,
    date_column="date",
    series_id_column="series_id",
)

print("Multi-series metrics:")
display(result_multi.metrics)
print(f"\nPredictions shape: {result_multi.predictions.shape}")
display(result_multi.predictions.head(10))

100%|██████████| 7/7 [00:00<00:00, 27.77it/s]

Multi-series metrics:


,levels,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,store_A,10.455031,502.869705,12.793333,0.106383
1,store_B,12.552790,525.733091,12.585153,0.059177
2,store_C,10.610921,413.138422,14.311675,0.074590
3,average,11.206247,480.580406,13.230054,0.080050
4,weighted_average,11.206247,480.580406,13.230054,0.080050
5,pooling,11.206247,480.580406,13.152509,0.080050



Predictions shape: (129, 3)


,level,fold,pred
2022-06-07,store_A,0,153.896112
2022-06-07,store_B,0,153.896112
2022-06-07,store_C,0,153.896112
2022-06-08,store_A,0,153.896112
2022-06-08,store_B,0,153.896112
2022-06-08,store_C,0,153.896112
2022-06-09,store_A,0,153.896112
2022-06-09,store_B,0,153.896112
2022-06-09,store_C,0,153.896112
2022-06-10,store_A,0,153.896112


---
## 5. CV with Gap (Simulating Deployment Delay)

Use `gap` to simulate scenarios where predictions are needed N steps into the future
but there's a delay between data collection and forecast deployment.

In [15]:
cv_gap, explanation = assistant.create_cv(
    profile, plan,
    gap=3,  # 3-day delay between training end and forecast start
)

print(explanation)

result_gap = assistant.backtest(
    data=df_single,
    target="sales",
    cv=cv_gap,
    date_column="date",
)

print("\nMetrics (with 3-day gap):")
display(result_gap.metrics)

Using 70% of data (255 observations) for initial training, expanding window, refit every fold, 14-step horizon, 8 folds, gap of 3 observations.


100%|██████████| 8/8 [00:00<00:00, 14.26it/s]


Metrics (with 3-day gap):


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,3.164613,14.075992,1.075357,0.033884


---
## 6. Refit Every N Folds

Pass `refit=int` to refit the model every N folds (balances accuracy vs speed).

In [16]:
cv_refit_n, explanation = assistant.create_cv(
    profile, plan,
    refit=3,  # Refit every 3 folds
)

print(explanation)

result_refit_n = assistant.backtest(
    data=df_single,
    target="sales",
    cv=cv_refit_n,
    date_column="date",
)

print("\nMetrics (refit every 3 folds):")
display(result_refit_n.metrics)

Using 70% of data (255 observations) for initial training, expanding window, refit every 3 folds, 14-step horizon, 8 folds.


100%|██████████| 8/8 [00:00<00:00, 46.04it/s]


Metrics (refit every 3 folds):


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,3.177969,16.290835,1.088489,0.033826


In [17]:
result_refit_n.explanation

'Using 70% of data (255 observations) for initial training, expanding window, refit every 3 folds, 14-step horizon, 8 folds. Results — mean_absolute_error: 3.1780, mean_squared_error: 16.2908, mean_absolute_scaled_error: 1.0885, mean_absolute_percentage_error: 0.0338.'

---
## 7. LLM-Assisted CV Configuration

When an LLM is configured, describe your deployment scenario in natural language.
The LLM maps business requirements to `TimeSeriesFold` parameters.

**Note:** Requires `llm` to be configured (uncomment and set your provider).

In [18]:
# Uncomment to use LLM-assisted CV:
#
# assistant_llm = ForecastingAssistant(llm="openai:gpt-4o-mini")
#
# cv_llm, explanation = assistant_llm.create_cv(
#     profile, plan,
#     prompt=(
#         "We retrain our model weekly in production. "
#         "There's a 2-day delay between data collection and deployment. "
#         "We want to simulate this scenario realistically."
#     ),
# )
#
# print(explanation)
# print(f"\n  refit:   {cv_llm.refit}")
# print(f"  gap:     {cv_llm.gap}")
# print(f"  stride:  {cv_llm.fold_stride}")

---
## 8. Explaining Results with `ask()`

Pass `backtest_result` to `ask()` to get LLM-powered analysis of your backtesting results.

**Note:** Requires `llm` to be configured.

In [19]:
# Uncomment to use ask() with backtest results:
#
# assistant_llm = ForecastingAssistant(llm="openai:gpt-4o-mini")
#
# answer = assistant_llm.ask(
#     prompt="Are the backtest metrics good? What could be improved?",
#     backtest_result=result,
# )
#
# print(answer.explanation)

---
## 9. Direct Use of `TimeSeriesFold`

Since `create_cv()` returns a standard skforecast `TimeSeriesFold`, you can also
construct one manually and pass it directly to `backtest()`.

In [20]:
from skforecast.model_selection import TimeSeriesFold

# Fully manual CV configuration
cv_manual = TimeSeriesFold(
    steps=14,
    initial_train_size=300,
    refit=False,
    fixed_train_size=True,
    gap=0,
)

result_manual = assistant.backtest(
    data=df_single,
    target="sales",
    cv=cv_manual,
    date_column="date",
)

print("Metrics (manual TimeSeriesFold):")
display(result_manual.metrics)

Information of folds
--------------------
Number of observations used for initial training: 300
Number of observations used for backtesting: 65
    Number of folds: 5
    Number skipped folds: 0 
    Number of steps per fold: 14
    Number of steps to exclude between last observed data (last window) and predictions (gap): 0
    Last fold only includes 9 observations.

Fold: 0
    Training:   0 -- 299  (n=300)
    Validation: 300 -- 313  (n=14)
Fold: 1
    Training:   No training in this fold
    Validation: 314 -- 327  (n=14)
Fold: 2
    Training:   No training in this fold
    Validation: 328 -- 341  (n=14)
Fold: 3
    Training:   No training in this fold
    Validation: 342 -- 355  (n=14)
Fold: 4
    Training:   No training in this fold
    Validation: 356 -- 364  (n=9)



100%|██████████| 5/5 [00:00<00:00, 811.40it/s]

Metrics (manual TimeSeriesFold):


,mean_absolute_error,mean_squared_error,mean_absolute_scaled_error,mean_absolute_percentage_error
0,4.415222,27.100297,1.524161,0.045997


---
## 10. Comparing CV Strategies

Compare metrics across different CV configurations to understand their impact.

In [21]:
comparison = pd.DataFrame({
    "Strategy": [
        "Default (expanding, refit)",
        "Fixed window, no refit",
        "3-day gap",
        "Refit every 3 folds",
    ],
    "MAE": [
        result.metrics.iloc[0, 0],
        result_custom.metrics.iloc[0, 0],
        result_gap.metrics.iloc[0, 0],
        result_refit_n.metrics.iloc[0, 0],
    ],
})

print("CV Strategy Comparison:")
display(comparison.sort_values("MAE"))

CV Strategy Comparison:


,Strategy,MAE
1,"Fixed window, no refit",2.946318
0,"Default (expanding, refit)",2.992479
2,3-day gap,3.164613
3,Refit every 3 folds,3.177969
